# Crate Music Database - Initial Exploration

This notebook provides an initial exploration of the Crate music database.

In [3]:
import sys
from pathlib import Path

# Add the src directory to the path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from crate_analysis import Database

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set up plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Connect to Database

In [4]:
# Connect to the database
db = Database()
print(f"Connected to database at: {db.db_path}")

Connected to database at: /Users/pooks/Dev/crate/data/music_kb.sqlite


## Database Schema Overview

In [5]:
# Get all tables
tables = db.get_tables()
print(f"Total tables: {len(tables)}\n")
tables

Total tables: 16



,name
0,artist_fact_plays
1,artists_masters
2,audit_log
3,effect_sql_migrations
4,entities_fts
5,entities_fts_config
6,entities_fts_content
7,entities_fts_data
8,entities_fts_docsize
9,entities_fts_idx


In [6]:
# Get row counts for each table
table_stats = []
for table_name in tables['name']:
    try:
        count = db.get_row_count(table_name)
        table_stats.append({'table': table_name, 'row_count': count})
    except Exception as e:
        print(f"Error getting count for {table_name}: {e}")

table_stats_df = pd.DataFrame(table_stats).sort_values('row_count', ascending=False)
table_stats_df

,table,row_count
11,master_relations,32640646
13,mb_master_lookup,26276365
10,fact_plays,2193187
0,artist_fact_plays,1785586
4,entities_fts,679814
6,entities_fts_content,679814
8,entities_fts_docsize,679814
2,audit_log,12563
9,entities_fts_idx,6637
7,entities_fts_data,5763


## Explore Fact Plays Table

In [7]:
# Get schema for fact_plays table
db.get_table_info('fact_plays')

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,1,None,0
1,1,airdate,TEXT,1,None,0
2,2,show,INTEGER,1,None,0
3,3,show_uri,TEXT,1,None,0
4,4,image_uri,TEXT,0,None,0
5,5,thumbnail_uri,TEXT,0,None,0
6,6,song,TEXT,0,None,0
7,7,track_id,TEXT,0,None,0
8,8,recording_id,TEXT,0,None,0
9,9,artist,TEXT,0,None,0


In [8]:
# Get sample rows
plays_sample = db.get_table_sample('fact_plays', 10)
plays_sample

,id,airdate,show,show_uri,image_uri,thumbnail_uri,song,track_id,recording_id,artist,artist_ids,album,release_id,release_group_id,labels,label_ids,release_date,rotation_status,is_local,is_request,is_live,comment,play_type,created_at,updated_at
0,3518527,2025-06-25T01:49:16-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia800307.us.archive.org/26/items/mbid-...,https://dn721508.ca.archive.org/0/items/mbid-a...,How I Became a Madman,3518527,2e89c6e9-540a-4ab6-ba65-1607d8a9fc3c,Ami Taf Ra feat. Kamasi Washington,"[""c7a3e868-c6d8-4512-a5ef-f6cbe42899b0"",""0e0b6...",The Prophet and the Madman,abc2013c-b852-416d-967d-d6a3f21780cf,f50a44d8-ed21-4519-9293-b960b0d7501f,"[""Brainfeeder""]","[""20b3d6f9-9086-48d9-802f-5f808456a0ef""]",2025-08-22,Medium,0,0,0,"North African, LA-based singer-songwriter Ami ...",trackplay,2025-06-25T09:16:30.253Z,2025-06-25T09:16:30.253Z
1,3518526,2025-06-25T01:46:50-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia601909.us.archive.org/3/items/mbid-5...,https://ia801909.us.archive.org/3/items/mbid-5...,Ain’t No Mountain High Enough,3518526,b04b5028-da5b-4f05-bfe3-4f8f80396d98,Marvin Gaye & Tammi Terrell,"[""afdb7919-059d-43c1-b668-ba1d265e7e42"",""ce458...",Hitsville USA: The Motown Singles Collection 1...,55e12b15-e080-42a8-af93-4301e3487a45,ebc5c8a7-5537-34b2-a7ad-a2bc50ba4a31,"[""Motown""]","[""8e479e57-ef44-490c-b75d-cd28df89bf1b""]",1992-11-03,None,0,0,0,The extraordinary Tammi Terrell died just befo...,trackplay,2025-06-25T09:16:30.254Z,2025-06-25T09:16:30.254Z
2,3518525,2025-06-25T01:42:30-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia801509.us.archive.org/34/items/mbid-...,https://ia601509.us.archive.org/34/items/mbid-...,Turn Me Around,3518525,831ddcea-19a5-4588-86d7-b093c4df9376,Mavis Staples & Bonnie Raitt,"[""0f0da09c-3940-4cb5-879f-4cea28907810"",""04f57...",I'll Take You There: An All-Star Concert Celeb...,950ec5b2-8dc3-41c3-beff-301c2eaa1588,038454a4-3998-4b42-8d5e-09af2fbea082,"[""Blackbird Presents""]","[""30ec5ddd-d62a-4167-aae7-89845b21ae1d""]",2017-06-02,Library,0,0,0,From Mavis Staples: I’ll Take You There — An A...,trackplay,2025-06-25T09:16:30.254Z,2025-06-25T09:16:30.254Z
3,3518524,2025-06-25T01:39:11-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia600109.us.archive.org/22/items/mbid-...,https://ia600109.us.archive.org/22/items/mbid-...,Lovers’ Holiday,3518524,cb65609c-6a14-45cf-a178-765be6354bb3,Durand Jones & The Indications,"[""5da8d9b1-af89-43a3-a519-8e32ec21e7f5""]",Flowers,8244e6f2-77f6-4a33-9482-4b6ee6cdc84d,ef7063f4-5f5c-49ea-a1b8-24a4b5d66351,"[""Dead Oceans""]","[""f70f950f-2587-4f85-a5c7-b483a47bd2e9""]",2025-06-27,Light,0,0,0,October 28th at Showbox SoDo\nhttps://www.yout...,trackplay,2025-06-25T09:16:30.255Z,2025-06-25T09:16:30.255Z
4,3518522,2025-06-25T01:33:03-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://dn721305.ca.archive.org/0/items/mbid-f...,https://dn721305.ca.archive.org/0/items/mbid-f...,Down to be wrong,3518522,885e1f40-5757-437a-a2d2-c096fa5425c1,HAIM,"[""aef06569-098f-4218-a577-b413944d9493""]",I quit,f8d6b4d8-8d8f-4c83-8f24-a6413dd3885a,86fdc55b-464f-4f6a-bcf7-86acf2964e87,"[""Columbia""]","[""011d1192-6f65-45bd-85c4-0400dd45693e""]",2025-06-20,Medium,0,0,0,HAIM will be on tour in support of their fourt...,trackplay,2025-06-25T09:16:30.255Z,2025-06-25T09:16:30.255Z
5,3518521,2025-06-25T01:29:19-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia800109.us.archive.org/13/items/mbid-...,https://ia600109.us.archive.org/13/items/mbid-...,Devil Gate Drive,3518521,ffbeedd6-dbe9-4301-8b33-48c0582f8cda,Suzi Quatro,"[""2c16cb3f-e85f-4158-889f-ffc038f5792d""]",Quatro,11791ee9-cf6d-4501-b7e3-242393b66145,28c577db-381f-36c4-a3e4-abf18c1828cc,"[""RAK""]","[""80c3ece5-2746-4364-8b34-f5ef2b983e21""]",1974-01-01,Library,0,0,0,This song describes a trip to Devil Gate Drive...,trackplay,2025-06-25T09:16:30.255Z,2025-06-25T09:16:30.255Z
6,351

In [9]:
# Get basic statistics about plays
plays_stats = db.query("""
    SELECT 
        COUNT(*) as total_plays,
        COUNT(DISTINCT artist) as unique_artists,
        COUNT(DISTINCT album) as unique_albums,
        COUNT(DISTINCT song) as unique_songs,
        MIN(airdate) as earliest_play,
        MAX(airdate) as latest_play
    FROM fact_plays
""")
plays_stats

,total_plays,unique_artists,unique_albums,unique_songs,earliest_play,latest_play
0,2193187,165294,252449,454619,2007-01-17T18:57:19-08:00,2025-11-11T10:45:29-08:00


## Top Artists Analysis

In [10]:
# Top 20 most played artists
top_artists = db.query("""
    SELECT 
        artist,
        COUNT(*) as play_count,
        COUNT(DISTINCT song) as unique_songs,
        COUNT(DISTINCT album) as unique_albums
    FROM fact_plays
    WHERE artist IS NOT NULL
    GROUP BY artist
    ORDER BY play_count DESC
    LIMIT 20
""")
top_artists

,artist,play_count,unique_songs,unique_albums
0,Radiohead,6006,422,192
1,LCD Soundsystem,5341,209,137
2,David Bowie,5253,522,250
3,The Cure,4817,296,134
4,The Clash,4283,276,140
5,Beck,4038,287,142
6,Arcade Fire,3861,154,55
7,Pixies,3716,190,107
8,Spoon,3451,202,75
9,Prince,3334,609,376


In [11]:
# Visualize top artists
fig = px.bar(top_artists, x='artist', y='play_count', 
             title='Top 20 Most Played Artists',
             labels={'play_count': 'Number of Plays', 'artist': 'Artist'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Temporal Analysis

In [12]:
# Plays over time (daily)
plays_over_time = db.query("""
    SELECT 
        DATE(airdate) as date,
        COUNT(*) as play_count
    FROM fact_plays
    GROUP BY DATE(airdate)
    ORDER BY date
""")
plays_over_time['date'] = pd.to_datetime(plays_over_time['date'])
plays_over_time.head()

,date,play_count
0,2007-01-18,281
1,2007-01-19,331
2,2007-01-20,339
3,2007-01-21,325
4,2007-01-22,297


In [13]:
# Plot plays over time
fig = px.line(plays_over_time, x='date', y='play_count',
              title='Daily Play Count Over Time',
              labels={'play_count': 'Number of Plays', 'date': 'Date'})
fig.show()

## Explore Relationships Table

In [14]:
# Check if master_relations table exists
if 'master_relations' in tables['name'].values:
    # Get schema
    print("Schema:")
    display(db.get_table_info('master_relations'))
    
    # Get sample
    print("\nSample rows:")
    display(db.get_table_sample('master_relations', 10))
    
    # Get relationship type distribution
    print("\nRelationship types:")
    rel_types = db.query("""
        SELECT predicate, COUNT(*) as count
        FROM master_relations
        GROUP BY predicate
        ORDER BY count DESC
    """)
    display(rel_types)
else:
    print("master_relations table not found")

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,subject_id,TEXT,1,None,0
1,1,subject_type,TEXT,1,None,0
2,2,subject_name,TEXT,0,None,0
3,3,predicate,TEXT,1,None,0
4,4,object_id,TEXT,1,None,0
5,5,object_type,TEXT,1,None,0
6,6,object_name,TEXT,0,None,0
7,7,attribute_type,TEXT,0,None,0
8,8,source,TEXT,1,None,0
9,9,kexp_play_id,INTEGER,0,None,0



Sample rows:


,subject_id,subject_type,subject_name,predicate,object_id,object_type,object_name,attribute_type,source,kexp_play_id,updated_at,created_at
0,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,d27fdd10-e1df-4208-8e2e-62187866246f,recording,Alright (radio version),,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
1,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,16a4347d-1ed8-4b73-b372-9a373d156c41,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
2,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,47985ebe-998e-4198-878d-c241258d2c3a,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
3,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,789d456b-f206-4804-bd6e-ddbe40fb6e3f,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
4,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,f9f77ccc-fd3b-4b1c-af98-be5f708b25cd,recording,Na Na Na Na,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
5,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,7ac765b7-cb48-4cd0-af23-f0e5b1f0a8b5,recording,Scalp Dem,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
6,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,producer,995d7811-c420-4066-b9bb-b53cb44cfc3c,recording,Dolly My Baby,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
7,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,d89b9792-7135-44ca-a3cd-da8556326bd9,recording,Fly,additional,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
8,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,16a4347d-1ed8-4b73-b372-9a373d156c41,recording,Fly,additional,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
9,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,56a3b1b1-1210-49a2-abf1-0d53571de1ed,recording,The Don of Dons (Put de Ting Pon Dem),,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12



Relationship types:


,predicate,count
0,instrument,6576850
1,vocal,2633697
2,producer,2244680
3,composer,2088189
4,has_artist,1743633
...,...,...
108,adapter,97
109,named after release group,87
110,video copyright,59
111,named after label,16


## Custom Analysis

Add your own queries and analysis below:

In [15]:
# Your custom queries here

In [16]:
# Don't forget to close the connection when done
db.close()